# Greenness and land cover
## Mexicali Urban Liveability Index — `WP04_greenness_and_land_cover`

**Lead:** TBC
**Indicators assigned:** 5
**Schema version:** 1.0.0

Vegetation quantity, canopy cover, vegetation mix and greenness from satellite imagery and municipal tree data.

In an arid city, distinguish irrigated urban vegetation from desert background; a raw NDVI threshold calibrated elsewhere will not transfer. Coordinate with WP06, which holds land use type (#405) -- land *cover* from imagery and land *use* from the municipal classification are different things and should not be conflated.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP04_greenness_and_land_cover'
        NOTEBOOK = 'notebooks/04_greenness_and_land_cover.ipynb'

---
## Your indicators (5)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 385 — Vegetation

`vegetation` · *Ambient Environment · Resources · Land · Vegetation*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Vegetation as a land use type is the optimal land category for human settlement, directly supporting liveability and wellbeing through ecosystem services and green space provision.
- **Adapted from:** #28: Chen (2023), ‘Evaluation of the livability of arid urban environments under global warming: A multi-parameter approach’

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_385) at any time to list what is
# still outstanding.
meta_385 = uli.metadata_stub(385, analyst=ANALYST)

# meta_385['rationale']['statement'] = """..."""
# meta_385['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_385['rationale']['arid_context'] = '...'
# meta_385['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_385['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_385)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_385 = 'grid_100m'
METHOD_385 = 'population_weighted_mean'

native_385 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_385 = uli.harmonise(
    native_385,
    native_scale=NATIVE_SCALE_385,
    method=METHOD_385,
)
results_385 = uli.label(
    harmonised_385,
    meta_385,
    measure_id='vegetation__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_385, meta_385))
# uli.write_indicator(results_385, meta_385)

### 135 — NDVI

`ndvi` · *Built Environment · Ambient · Vegetation · NDVI*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** The Normalised Difference Vegetation Index (NDVI) is used to track city-wide greenness, which supports mental health and environmental sustainability.
- **Adapted from:** #13: Alderton (2021), ‘Measuring and monitoring liveability in a low-to-middle income country: a proof-of-concept for Bangkok, Thailand and lessons from an international partnership’

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_135) at any time to list what is
# still outstanding.
meta_135 = uli.metadata_stub(135, analyst=ANALYST)

# meta_135['rationale']['statement'] = """..."""
# meta_135['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_135['rationale']['arid_context'] = '...'
# meta_135['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_135['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_135)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_135 = 'grid_100m'
METHOD_135 = 'population_weighted_mean'

native_135 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_135 = uli.harmonise(
    native_135,
    native_scale=NATIVE_SCALE_135,
    method=METHOD_135,
)
results_135 = uli.label(
    harmonised_135,
    meta_135,
    measure_id='ndvi__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_135, meta_135))
# uli.write_indicator(results_135, meta_135)

### 378 — Vegetation coverage: Canopy

`vegetation_coverage_canopy` · *Built Environment · Ambient · Vegetation · Vegetation coverage: Canopy*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Urban tree canopy coverage mitigates local temperatures and provides shade and aesthetic value that support mental wellbeing and encourage physical activity in public spaces.
- **Adapted from:** #22: Alderton (2019), ‘What is the meaning of urban liveability for a city in a low-to-middle-income country? Contextualising liveability for Bangkok, Thailand’

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_378) at any time to list what is
# still outstanding.
meta_378 = uli.metadata_stub(378, analyst=ANALYST)

# meta_378['rationale']['statement'] = """..."""
# meta_378['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_378['rationale']['arid_context'] = '...'
# meta_378['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_378['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_378)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_378 = 'grid_100m'
METHOD_378 = 'population_weighted_mean'

native_378 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_378 = uli.harmonise(
    native_378,
    native_scale=NATIVE_SCALE_378,
    method=METHOD_378,
)
results_378 = uli.label(
    harmonised_378,
    meta_378,
    measure_id='vegetation_coverage_canopy__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_378, meta_378))
# uli.write_indicator(results_378, meta_378)

### 388 — Vegetation mix

`vegetation_mix` · *Built Environment · Ambient · Vegetation · Vegetation mix*

- **Lenses to deliver:** diversity
- **Draft rationale (rewrite this):** A variety of urban vegetation supports biodiversity and ecological health, while providing aesthetic and psychological benefits that enhance the living environment.
- **Adapted from:** #16: Kashi (2025), ‘Spatial analysis and ranking of urban districts based on a comprehensive livability approach: the case of Tehran’

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_388) at any time to list what is
# still outstanding.
meta_388 = uli.metadata_stub(388, analyst=ANALYST)

# meta_388['rationale']['statement'] = """..."""
# meta_388['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_388['rationale']['arid_context'] = '...'
# meta_388['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_388['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_388)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_388 = 'grid_100m'
METHOD_388 = 'population_weighted_mean'

native_388 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_388 = uli.harmonise(
    native_388,
    native_scale=NATIVE_SCALE_388,
    method=METHOD_388,
)
results_388 = uli.label(
    harmonised_388,
    meta_388,
    measure_id='vegetation_mix__diversity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_388, meta_388))
# uli.write_indicator(results_388, meta_388)

### 389 — Vegetation percent

`vegetation_percent` · *Ambient Environment · Ambient · Vegetation · Vegetation percent*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Mean vegetation coverage across the city helps mitigate the urban heat island effect and provides psychological health benefits through exposure to nature.
- **Adapted from:** #13: Alderton (2021), ‘Measuring and monitoring liveability in a low-to-middle income country: a proof-of-concept for Bangkok, Thailand and lessons from an international partnership’

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_389) at any time to list what is
# still outstanding.
meta_389 = uli.metadata_stub(389, analyst=ANALYST)

# meta_389['rationale']['statement'] = """..."""
# meta_389['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_389['rationale']['arid_context'] = '...'
# meta_389['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_389['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_389)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_389 = 'grid_100m'
METHOD_389 = 'population_weighted_mean'

native_389 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_389 = uli.harmonise(
    native_389,
    native_scale=NATIVE_SCALE_389,
    method=METHOD_389,
)
results_389 = uli.label(
    harmonised_389,
    meta_389,
    measure_id='vegetation_percent__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_389, meta_389))
# uli.write_indicator(results_389, meta_389)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()